In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_debug_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe069.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_debug_export.log


In [59]:
input_phased_path = "data/mt/ukb_wes_200k_annotated_chr21.mt"
input_unphased_path = "data/mt/ukb_wes_200k_annotated_chr21_singletons.mt"


In [41]:


OUT = '/well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_qc_sex.ht'
OUT_TXT = '/well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_qc_sex.txt.bgz'
ht = hl.read_table('/well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_imputesex.ht')
ht_sexcheck_samples = hl.import_table('/well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_sexcheck.remove.sample_list', no_header = True, key = 'f0')
ht = ht.annotate(pass_sexcheck = ~hl.is_defined(ht_sexcheck_samples[ht.key]))
ht = ht.filter(ht.pass_sexcheck)
ht.write(OUT, overwrite=True)
ht.export(OUT_TXT)


#ht = ht.annotate(impute_sex = impute_sex_annotations[ht.key])

#ht.describe()
#ht = mt_before.annotate_cols(imputesex = impute_sex_annotations[mt_before.col_key])

#ht = ht.filter_cols()
#tb.show()
#ht.is_female.



2021-11-05 11:56:16 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
2021-11-05 11:56:16 Hail: INFO: Coerced sorted dataset
2021-11-05 11:56:19 Hail: INFO: wrote table with 487909 rows in 16 partitions to /well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_qc_sex.ht
2021-11-05 11:56:19 Hail: INFO: Coerced sorted dataset
2021-11-05 11:56:28 Hail: INFO: merging 16 files totalling 177.7M...
2021-11-05 11:56:29 Hail: INFO: while writing:
    /well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_qc_sex.txt.bgz
  merge time: 286.064ms


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

(60590, 185506)
(60590, 102046)
(60590, 102046)


In [88]:
females = get_females_expr(mt.s)
males = ~females

In [91]:
print(sum(filter(None, females.collect())))
print(sum(filter(None, males.collect())))

102046
83380


In [101]:
def get_sex_qc_path():
    r''' '''
    return '/well/lindgren/flassen/ressources/ukb/sex/nov21/04_ukb_imp_500k_qc_sex.ht'

def get_females_expr(expr_sample_id):

    path = get_sex_qc_path()
    ht = hl.read_table(path)
    ht = ht.filter(hl.is_defined(ht.is_female))
    return ht[expr_sample_id].is_female

def filter_to_sex(mt, sex = 'females'):
    females = get_females_expr(mt.s)
    if sex in 'males':
        return mt.filter_cols(~females)
    elif sex in 'females':
        return mt.filter_cols(females)
    else:
        raise TypeError(f"param sex must be either 'females' or 'males'")

In [102]:
mt = qc.get_table(input_path=input_phased_path, input_type='mt') 
print(mt.count())
print(filter_to_sex(mt, sex = 'females').count())
print(filter_to_sex(mt, sex = 'males').count())

(60590, 185506)
(60590, 102046)
(60590, 83380)


In [40]:
mt = qc.get_table(input_path=input_phased_path, input_type='mt') 
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt')

In [11]:
consequence_annotations = hl.read_table('data/vep/hail/ukb_wes_200k_chr21_vep.ht')
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 
mt2 = mt2.annotate_rows(consequence = consequence_annotations[mt2.row_key])

In [12]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
mt2 = mt2.explode_rows(mt2.consequence.vep.worst_csq_by_gene_canonical)    

In [14]:
mt1.consequence.vep.worst_csq_for_variant_canonical.gene_id

<StringExpression of type str>

In [17]:
category_annotation_mt1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_for_variant_canonical, mt1.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = category_annotation_mt1)
category_annotation_mt2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_for_variant_canonical, mt2.consequence.dbnsfp, use_loftee = False)
mt2 = mt1.annotate_rows(consequence_category = category_annotation_mt2)




In [19]:
expr_gene_id = mt1.consequence.vep.worst_csq_for_variant_canonical.gene_id
analysis.count_urv_by_genes(mt1, expr_gene_id)


In [21]:
ht = hl.read_table('/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/200k/phenotypes.ht')

In [28]:
TRANCHE = '200k'
SEXCHECK_LIST = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_sexcheck.remove.sample_list'
PHENOTYPES_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/' + TRANCHE + '/phenotypes.ht'
IMPUTESEX_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_imputesex.ht'

sample_annotations = hl.read_table(PHENOTYPES_TABLE)
impute_sex_annotations = hl.read_table(IMPUTESEX_TABLE)
ht_sexcheck_samples = hl.import_table(SEXCHECK_LIST, no_header=True, key='f0')



2021-11-04 10:14:26 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)


In [53]:

#impute_sex_annotations.count()

mt.count()

#impute_sex_annotations.aggregate(hl.agg.sum(~hl.is_defined(impute_sex_annotations.is_female)))
#impute_sex_annotations.count()
#mt = mt.annotate_cols(is_female = impute_sex_annotations[mt.col_key].is_female)
#mt = mt.filter_cols(~hl.is_defined(ht_sexcheck_samples[mt.col_key]))
#x = mt.aggregate_cols(hl.agg.sum(~hl.is_defined(mt.is_female)))
#x

(60590, 185427)

In [61]:
# load imputed sex table
IMPUTESEX_TABLE = '/well/lindgren/UKBIOBANK/dpalmer/wes_' + TRANCHE + '/ukb_wes_qc/data/samples/04_imputesex.ht'
print(impute_sex_annotations.count())
print(impute_sex_annotations.aggregate(hl.agg.sum(~hl.is_defined(impute_sex_annotations.is_female))))


190792
0


In [62]:
# load phased data
mt = qc.get_table(input_path=input_phased_path, input_type='mt') 
print(mt.count())

(60590, 185506)


In [72]:
# subset to final samples
final_sample_path='/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list'
final_samples = hl.import_table(final_sample_path, delimiter = ',')


2021-11-04 10:28:15 Hail: WARN: Found 92 duplicate columns. Mangled columns follows:
  '0' -> '0_1'
  'F' -> 'F_1'
  'Batch 1' -> 'Batch 1_1'
  '0.991429222350438' -> '0.991429222350438_1'
  '9090824' -> '9090824_1'
  '78589' -> '78589_1'
  '0' -> '0_2'
  '9056681' -> '9056681_1'
  '20311' -> '20311_1'
  '13832' -> '13832_1'
  ...
2021-11-04 10:28:15 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127


In [68]:
# annotate with imputed sex
mt = mt.annotate_cols(is_female = impute_sex_annotations[mt.col_key].is_female)
print(mt.aggregate_cols(hl.agg.sum(~hl.is_defined(mt.is_female))))
mt.filter_cols(~hl.is_defined(mt.is_female)).show()

8207


,,,,,,,,,,,,,
,,'5886032','3100554','3173663','4889080','2422881','3466501','3609838','2132135','5793388','1492165','5798839','2703312'
locus,alleles,GT,GT,GT,GT,GT,GT,GT,GT,GT,GT,GT,GT
locus<GRCh38>,array<str>,call,call,call,call,call,call,call,call,call,call,call,call
chr21:10413617,"[""C"",""T""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413618,"[""G"",""A""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413629,"[""C"",""T""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413634,"[""T"",""G""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413636,"[""G"",""T""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413653,"[""A"",""G""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0
chr21:10413662,"[""C"",""T""]",0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0,0|0


In [9]:
#by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
#by_gene_annotation2 = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_by_gene_canonical, mt2.consequence.dbnsfp, use_loftee = False)

In [10]:
#mt1_subset = mt1.annotate_rows(consequence_category = by_gene_annotation1)    
#mt2_subset = mt2.annotate_rows(consequence_category = by_gene_annotation2) 

In [12]:
lst = by_gene_annotation1.collect()

In [13]:
from collections import Counter

In [14]:
Counter(lst)

Counter({'non_coding': 342230,
         'other_missense': 97576,
         'damaging_missense': 9812,
         'synonymous': 55482,
         'ptv': 7278,
         None: 11153})

In [15]:
mt1_subset = mt1_subset.filter_rows(mt1_subset.consequence_category == 'ptv')
mt2_subset = mt2_subset.filter_rows(mt2_subset.consequence_category == 'ptv')
mt1_subset.count()

(7278, 185506)

In [16]:
mt = analysis.gene_csqs_calc_pKO_pseudoSNP(mt1_subset, mt2_subset, 16)

2021-11-01 18:26:05 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


In [18]:
print(mt.count())

(790, 185506)


2021-11-01 18:46:24 Hail: INFO: Ordering unsorted dataset with network shuffle


In [17]:
print(mt.aggregate_entries(hl.agg.sum(~hl.is_defined(mt.DS))))

2021-11-01 18:28:22 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:30:29 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:32:33 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:46:04 Hail: INFO: Ordering unsorted dataset with network shuffle


0


In [19]:
missing = 0.05
if missing:
    mt1 = qc.filter_min_missing(mt1, float(missing))
    mt2 = qc.filter_min_missing(mt2, float(missing))
    
af_max = 0.50
if af_max:
    mt1 = qc.filter_max_af(mt1, float(af_max))

mt = analysis.gene_csqs_calc_pKO_pseudoSNP(mt1_subset, mt2_subset, 16)
print(mt.count())
print(mt.aggregate_entries(hl.agg.sum(~hl.is_defined(mt.DS))))

2021-11-01 18:52:59 Hail: INFO: Ordering unsorted dataset with network shuffle


(790, 185506)


2021-11-01 18:55:08 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:57:13 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 18:59:16 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-11-01 19:12:44 Hail: INFO: Ordering unsorted dataset with network shuffle


0
